# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

## Imports

In [ ]:
from src.utils.video import open_video, get_frames, save_video, produce_detection_output_video
from src.tracking.motion_detection import MOG2_motion_detection, refine_blobs
from src.tracking.image_processing import opening_closing, normalize_illumination
from src.utils.visualization import show_image, show_images
from src.detection.yolo_detection import run_yolo_detection, yolo_to_detection_output

import os
import time
import logging
from ultralytics import YOLO

# Suppress YOLO logging
logging.getLogger("ultralytics").setLevel(logging.WARNING)
logging.getLogger("ultralytics.hub.utils").setLevel(logging.WARNING)

In [ ]:
MODELS_DIR = "models/"
CURRENT_CAMERA_ID = "cam_13"

## Path Definitions

In [ ]:
VIDEOS_DIR = "data/videos"
CAMERA_SETTINGS_DIR = "data/camera_settings"

# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json",
    },
}

## Open Video and Read Frames

In [ ]:
# Currently just one camera
cap = open_video(CAMERAS[CURRENT_CAMERA_ID]["video_path"])
frames_color, _ = get_frames(cap, max_frames=150)  # Load only the first 150 frames for testing

# Release the video capture object to free resources
cap.release()

show_images(frames_color)

## Using YOLOv11 for Player Detection
We could also use a deep learning-based object detection model like YOLOv11 to detect players in each frame. This can be done by loading a pre-trained YOLOv11 model and running inference on the frames to get bounding boxes for detected players.

In [ ]:
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)
model_path = os.path.join(MODELS_DIR, "yolo11m.pt")

model = YOLO(model_path)    # Load the YOLOv11m model

### Run YOLO Detection on All Frames

Use YOLOv8's built-in `.predict()` method to detect objects in each frame. The output will include bounding boxes, confidence scores, and class labels for each detected object.

**Output per frame:**
- Bounding boxes: `[x1, y1, x2, y2]` (pixel coordinates)
- Confidence scores: Detection confidence (0–1)
- Class labels: What was detected (e.g., "person", "sports_ball")

In [ ]:
print(f"Processing {len(frames_color)} frames with YOLO detection (player pass)...")
start_time = time.time()

# Pass 1 — players only at normal resolution.
# class_ids=[0] is COCO "person". After fine-tuning, swap to the new model's
# player/referee/goalkeeper IDs.
raw_player_results = run_yolo_detection(
    model, frames_color,
    conf_threshold=0.4,
    inference_size=640,
    class_ids=[0],
)

end_time = time.time()
print(f"Player pass complete")
print(f"Total player-pass detections: {sum(len(r.boxes) for r in raw_player_results)}")
print(f"Processing time: {end_time - start_time:.2f} seconds")


### Higher-Resolution Pass — Ball Only

The football is a small object (~10–20 px in match footage) and the standard 640-px inference grid struggles to localize it. We therefore run a **second** inference pass at higher `imgsz` (1280) with a lower confidence threshold.

Critically, this pass is **class-filtered to the ball only**. Running high-resolution inference for `person` would also surface tiny bench players and spectators in the stands — exactly what we don't want.


In [ ]:
print(f"Processing {len(frames_color)} frames with YOLO detection (ball pass)...")
start_time = time.time()

# Pass 2 — ball only at high resolution.
# class_ids=[32] is COCO "sports ball". After fine-tuning, swap to the new model's ball ID.
# The low conf_threshold trades false positives for recall; tune it on your footage.
raw_ball_results = run_yolo_detection(
    model, frames_color,
    conf_threshold=0.1,
    inference_size=1280,
    class_ids=[32],
)

end_time = time.time()
print(f"Ball pass complete")
print(f"Total ball-pass detections: {sum(len(r.boxes) for r in raw_ball_results)}")
print(f"Processing time: {end_time - start_time:.2f} seconds")


### Merge Player and Ball Detections

Combine both passes into a single `DetectionOutput`, frame by frame. After fine-tuning, only the `class_ids` lists in the two passes above need to change — the merge stays identical.


In [ ]:
from src.detection.schema import DetectionOutput, Frame_Detections

player_out = yolo_to_detection_output(
    raw_player_results, model,
    camera_id=CURRENT_CAMERA_ID, fps=25, source="yolo_v11m_pt",
)
ball_out = yolo_to_detection_output(
    raw_ball_results, model,
    camera_id=CURRENT_CAMERA_ID, fps=25, source="yolo_v11m_pt",
)

detection_output = DetectionOutput(
    source=player_out.source,
    camera_id=player_out.camera_id,
    fps=player_out.fps,
    frames=[
        Frame_Detections(frame_index=p.frame_index, detections=p.detections + b.detections)
        for p, b in zip(player_out.frames, ball_out.frames)
    ],
)

print(f"Merged detections: {sum(f.num_detections for f in detection_output.frames)} across {len(detection_output.frames)} frames")


## Inspect Merged Detections


In [ ]:
print(f"First frame detections: {detection_output.frames[0].num_detections} objects detected")
print(f"First frame detection details:")
for i, detection in enumerate(detection_output.frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={detection.class_name}, Confidence={detection.confidence:.2f}, BBox={detection.bbox}")

# Quick visual check using YOLO's built-in .plot() on the player pass.
show_images(raw_player_results, is_yolo=True)


In [ ]:
produce_detection_output_video(frames_color, ball_out, f"results/detection/{CURRENT_CAMERA_ID}/detections.mp4")